In [0]:
# Databricks Notebook: Scott Steiert - Apex Idaho Expansion & Competitive Intelligence Pipeline
# Project Focus: Ingesting regional market data to identify target acquisitions and competitive threats for Apex
# Target Save File: scott_steiert_apex_idaho_pipeline.py

import dlt
from pyspark.sql.functions import col, current_timestamp, input_file_name, regexp_replace, trim, lower, upper, when, lit

# --------------------------------------------------------------------
# BRONZE LAYER: Raw Data Ingestion via Auto Loader
# --------------------------------------------------------------------

@dlt.table(
    name="bronze_apex_market_intelligence",
    comment="Raw streaming text data (SEC filings, local trade registries) for LLM fine-tuning and context retrieval."
)
def bronze_apex_market_intelligence():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("cloudFiles.schemaLocation", "/mnt/apex-intel/checkpoints/text_schema")
        .load("/mnt/apex-intel/raw/competitor_text/*/*")
        .select(
            col("value").alias("raw_text"),
            input_file_name().alias("source_file"),
            current_timestamp().alias("ingestion_time"),
            lit("Scott Steiert").alias("analyst_assigned"),
            lit("Apex Service Partners").alias("target_enterprise")
        )
    )

@dlt.table(
    name="bronze_regional_telemetry",
    comment="Raw structured json streams containing distributor shipments, pricing indexes, and localized market volume."
)
def bronze_regional_telemetry():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", "/mnt/apex-intel/checkpoints/metrics_schema")
        .load("/mnt/apex-intel/raw/market_metrics/*.json")
        .select(
            col("*"),
            input_file_name().alias("source_file"),
            current_timestamp().alias("ingestion_time"),
            lit("Scott Steiert").alias("analyst_assigned")
        )
    )

# --------------------------------------------------------------------
# SILVER LAYER: Cleaning and LLM Text Optimization
# --------------------------------------------------------------------

@dlt.table(
    name="silver_apex_market_intelligence",
    comment="Cleaned unstructured data mapped to explicit geographic indicators for localized expansion analysis."
)
@dlt.expect_or_drop("has_content", "length(clean_text) > 15")
def silver_apex_market_intelligence():
    return (
        dlt.read_stream("bronze_apex_market_intelligence")
        # Strip trailing formatting breaks to avoid messing up downstream chunking algorithms
        .withColumn("clean_text", regexp_replace(col("raw_text"), r'[\r\n\t]+', " "))
        .withColumn("clean_text", trim(col("clean_text")))
        # Isolate Pacific Northwest / Idaho operational theaters to optimize context windows and API token costs
        .withColumn("idaho_theater_match", 
            col("clean_text").rlike("(?i)(idaho|boise|meridian|nampa|idaho falls|coeur d|twin falls)")
        )
        .select("target_enterprise", "analyst_assigned", "source_file", "clean_text", "idaho_theater_match", "ingestion_time")
    )

@dlt.table(
    name="silver_regional_telemetry",
    comment="Standardized metrics capturing revenue and baseline operational margins for regional market comparison."
)
@dlt.expect_or_drop("positive_revenue", "revenue >= 0")
def silver_regional_telemetry():
    return (
        dlt.read_stream("bronze_regional_telemetry")
        .withColumn("competitor_ticker", upper(trim(col("ticker"))))
        .withColumn("target_region", lower(trim(col("region"))))
        .withColumn("revenue", col("revenue").cast("double"))
        .withColumn("operating_margin", col("operating_margin").cast("double"))
        .withColumn("reporting_period", upper(trim(col("quarter"))))
        # Filter metrics specifically to the Pacific Northwest to build the comparative baseline against Apex expansion
        .filter(col("target_region") == "pacific northwest")
        .select("competitor_ticker", "reporting_period", "target_region", "revenue", "operating_margin", "analyst_assigned", "ingestion_time")
    )

# --------------------------------------------------------------------
# GOLD LAYER: Aggregations for Power BI/Tableau Dashboards
# --------------------------------------------------------------------

@dlt.table(
    name="gold_apex_competitor_benchmarks",
    comment="Final analytics table for Power BI reporting, establishing baseline competitor margins in the target territory."
)
def gold_apex_competitor_benchmarks():
    return (
        dlt.read("silver_regional_telemetry")
        .groupBy("reporting_period", "competitor_ticker", "analyst_assigned")
        .agg(
            {"revenue": "sum", "operating_margin": "avg"}
        )
        .withColumnRenamed("sum(revenue)", "total_regional_revenue")
        .withColumnRenamed("avg(operating_margin)", "avg_competitor_margin")
    )